# 🏥 MMedAgent-Lite – D1 : Pipeline VQA-RAD
**Projet** : MMedAgent-Lite DS1  
**Dataset** : VQA-RAD (Radiologie)  
**HuggingFace** : `flaviagiammarino/vqa-rad`

---
### Ce que fait ce notebook :
1. Vérification GPU / RAM
2. Installation des bibliothèques
3. Montage Google Drive
4. Téléchargement VQA-RAD
5. Exploration et statistiques complètes
6. Pipeline P1-Rad : CLAHE + resize + normalisation ImageNet
7. Visualisations pour le rapport
8. Sauvegarde preprocess_rad.py + rapport statistiques

## ÉTAPE 0 – Vérification GPU et RAM

In [ ]:
!nvidia-smi 2>/dev/null || echo 'Pas de GPU – mode CPU'
import os
print('RAM disponible :', os.popen('free -h').read())
print('Espace disque  :', os.popen('df -h /content').read())

In [ ]:
!pip install datasets pillow opencv-python-headless numpy matplotlib pandas scikit-learn tqdm -q
print('Installation terminée')

## ÉTAPE 1 – Montage Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

BASE_DIR = '/content/drive/MyDrive/MMedAgent-Lite'
os.makedirs(f'{BASE_DIR}/dataset', exist_ok=True)
os.makedirs(f'{BASE_DIR}/data/figures', exist_ok=True)
os.makedirs(f'{BASE_DIR}/data/splits', exist_ok=True)
os.makedirs(f'{BASE_DIR}/dataset/scripts', exist_ok=True)

print('Google Drive monté')
print('Structure créée :')
print(f'  {BASE_DIR}/dataset/')
print(f'  {BASE_DIR}/data/figures/')
print(f'  {BASE_DIR}/data/splits/')

## ÉTAPE 2 – Téléchargement VQA-RAD

In [ ]:
from datasets import load_dataset

DATASET_CACHE = f'{BASE_DIR}/dataset/vqa_rad_raw'

if os.path.exists(DATASET_CACHE):
    print('Dataset déjà en cache sur Drive, chargement local...')
    from datasets import load_from_disk
    vqa_rad = load_from_disk(DATASET_CACHE)
    print('VQA-RAD chargé depuis Drive !')
else:
    print('Téléchargement VQA-RAD (~0.3 Go)...')
    vqa_rad = load_dataset('flaviagiammarino/vqa-rad')
    print('Sauvegarde sur Drive pour éviter re-téléchargement...')
    vqa_rad.save_to_disk(DATASET_CACHE)
    print('VQA-RAD sauvegardé sur Drive !')

print(vqa_rad)

In [ ]:
# Inspection des colonnes
print('Colonnes disponibles :', vqa_rad['train'].column_names)
print()
print('--- Exemple [0] ---')
exemple = vqa_rad['train'][0]
for cle, valeur in exemple.items():
    if cle != 'image':
        print(f'  {cle} : {valeur}')
print(f'  image : {exemple["image"].size} mode={exemple["image"].mode}')

## ÉTAPE 3 – Exploration et Statistiques Complètes

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

def build_dataframe(split, split_name):
    """Construit un DataFrame avec statistiques pour un split VQA-RAD."""
    rows = []
    for i, ex in enumerate(tqdm(split, desc=f'Analyse {split_name}')):
        ans = ex['answer'].strip().lower()
        q   = ex['question'].strip()
        img = ex['image']
        rows.append({
            'idx'        : i,
            'question'   : q,
            'answer'     : ans,
            'q_len'      : len(q.split()),
            'a_len'      : len(ans.split()),
            'first_word' : q.split()[0].lower() if q else '',
            'is_yesno'   : ans in ['yes', 'no'],
            'img_width'  : img.size[0],
            'img_height' : img.size[1],
            'img_mode'   : img.mode,
            'img_ratio'  : img.size[0] / max(img.size[1], 1),
        })
    return pd.DataFrame(rows)

print('Construction des DataFrames...')
df_train = build_dataframe(vqa_rad['train'], 'train')
df_test  = build_dataframe(vqa_rad['test'],  'test')
df_all   = pd.concat([df_train, df_test], ignore_index=True)

print(f'\nDimensions train : {df_train.shape}')
print(f'Dimensions test  : {df_test.shape}')
print(f'Total            : {len(df_all)}')

In [ ]:
# Analyse des réponses yes / no / ouvertes
n_yes  = (df_train['answer'] == 'yes').sum()
n_no   = (df_train['answer'] == 'no').sum()
n_open = (~df_train['is_yesno']).sum()

print('=== Analyse des réponses (train) ===')
print(f'  Yes      : {n_yes} ({100*n_yes/len(df_train):.1f}%)')
print(f'  No       : {n_no} ({100*n_no/len(df_train):.1f}%)')
print(f'  Ouvertes : {n_open} ({100*n_open/len(df_train):.1f}%)')
print()
print('=== Statistiques questions ===')
print(f'  Longueur moy : {df_train["q_len"].mean():.2f} mots')
print(f'  Longueur max : {df_train["q_len"].max()} mots')
print()
print('=== Premiers mots (top 8) ===')
print(df_train['first_word'].value_counts().head(8).to_string())
print()
print('=== Statistiques images ===')
print(f'  Modes       : {df_train["img_mode"].value_counts().to_dict()}')
print(f'  Largeur moy : {df_train["img_width"].mean():.1f} px')
print(f'  Hauteur moy : {df_train["img_height"].mean():.1f} px')
print(f'  Larg min/max: {df_train["img_width"].min()} / {df_train["img_width"].max()} px')

## ÉTAPE 4 – Visualisations Statistiques

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('VQA-RAD – Statistiques Générales (D1)', fontsize=16, fontweight='bold')

# 1. Distribution yes / no / ouvertes
ax = axes[0, 0]
labels_pie = ['Yes', 'No', 'Ouvertes']
sizes_pie  = [n_yes, n_no, n_open]
colors_pie = ['#4CAF50', '#F44336', '#2196F3']
ax.pie(sizes_pie, labels=labels_pie, colors=colors_pie,
       autopct='%1.1f%%', startangle=90)
ax.set_title('Types de réponses (train)')

# 2. Longueur des questions
ax = axes[0, 1]
ax.hist(df_train['q_len'], bins=25, color='#9C27B0', edgecolor='white', alpha=0.85)
ax.axvline(df_train['q_len'].mean(), color='red', linestyle='--',
           label=f'Moy: {df_train["q_len"].mean():.1f}')
ax.set_xlabel('Nombre de mots')
ax.set_ylabel('Fréquence')
ax.set_title('Longueur des questions')
ax.legend()

# 3. Top mots initiaux
ax = axes[0, 2]
top_words = df_train['first_word'].value_counts().head(8)
ax.barh(top_words.index[::-1], top_words.values[::-1], color='#FF9800')
ax.set_xlabel('Fréquence')
ax.set_title('Top 8 premiers mots des questions')
for i, v in enumerate(top_words.values[::-1]):
    ax.text(v + 2, i, str(v), va='center', fontsize=9)

# 4. Distribution largeurs images
ax = axes[1, 0]
ax.hist(df_train['img_width'], bins=35, color='#00BCD4', edgecolor='white', alpha=0.85)
ax.axvline(df_train['img_width'].mean(), color='red', linestyle='--',
           label=f'Moy: {df_train["img_width"].mean():.0f}px')
ax.set_xlabel('Largeur (px)')
ax.set_ylabel('Fréquence')
ax.set_title('Distribution largeurs images')
ax.legend()

# 5. Distribution hauteurs images
ax = axes[1, 1]
ax.hist(df_train['img_height'], bins=35, color='#8BC34A', edgecolor='white', alpha=0.85)
ax.axvline(df_train['img_height'].mean(), color='red', linestyle='--',
           label=f'Moy: {df_train["img_height"].mean():.0f}px')
ax.set_xlabel('Hauteur (px)')
ax.set_ylabel('Fréquence')
ax.set_title('Distribution hauteurs images')
ax.legend()

# 6. Modes PIL
ax = axes[1, 2]
modes = df_train['img_mode'].value_counts()
ax.bar(modes.index, modes.values,
       color=['#E91E63', '#3F51B5', '#FF5722'][:len(modes)])
ax.set_xlabel('Mode PIL')
ax.set_ylabel("Nombre d'images")
ax.set_title("Modes d'images (PIL)")
for i, v in enumerate(modes.values):
    ax.text(i, v + 2, str(v), ha='center', fontsize=10)

plt.tight_layout()
fig_path = f'{BASE_DIR}/data/figures/stats_vqarad.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure sauvegardée : {fig_path}')

In [ ]:
# Distribution des catégories de questions (11 catégories VQA-RAD)
# Détection automatique par mot-clé du premier mot
category_map = {
    'is'      : 'Yes/No – Présence',
    'are'     : 'Yes/No – Présence',
    'does'    : 'Yes/No – Fonction',
    'do'      : 'Yes/No – Fonction',
    'was'     : 'Yes/No – Historique',
    'where'   : 'Localisation',
    'what'    : 'Identification',
    'which'   : 'Identification',
    'how'     : 'Quantification',
    'can'     : 'Yes/No – Possibilité',
}

def map_category(first_word):
    return category_map.get(first_word.lower(), 'Autre')

df_train['category'] = df_train['first_word'].apply(map_category)
cat_counts = df_train['category'].value_counts()

fig, ax = plt.subplots(figsize=(12, 5))
colors_cat = plt.cm.tab10(range(len(cat_counts)))
bars = ax.bar(cat_counts.index, cat_counts.values, color=colors_cat, edgecolor='white')
ax.set_title('VQA-RAD – Distribution des catégories de questions (train)', fontsize=13, fontweight='bold')
ax.set_ylabel('Nombre de Q&A')
ax.set_xlabel('Catégorie')
plt.xticks(rotation=30, ha='right')
for bar, v in zip(bars, cat_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, v + 5,
            f'{v}\n({100*v/len(df_train):.1f}%)',
            ha='center', fontsize=8)
plt.tight_layout()
fig_path = f'{BASE_DIR}/data/figures/categories_vqarad.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure sauvegardée : {fig_path}')

In [ ]:
# Exemples d'images VQA-RAD avec questions/réponses
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle('VQA-RAD – Exemples (haut: Yes/No | bas: Questions ouvertes)',
             fontsize=13, fontweight='bold')

# Yes/No
yesno_idx = df_train[df_train['is_yesno']].sample(4, random_state=42)['idx'].tolist()
for col, idx in enumerate(yesno_idx):
    ex = vqa_rad['train'][idx]
    axes[0, col].imshow(ex['image'], cmap='gray' if ex['image'].mode == 'L' else None)
    axes[0, col].axis('off')
    q = ex['question']
    q_wrap = '\n'.join([q[i:i+28] for i in range(0, len(q), 28)])
    color = '#4CAF50' if ex['answer'].lower() == 'yes' else '#F44336'
    axes[0, col].set_title(f'Q: {q_wrap}\nA: {ex["answer"]}',
                            fontsize=8, color=color)

# Ouvertes
open_idx = df_train[~df_train['is_yesno']].sample(4, random_state=42)['idx'].tolist()
for col, idx in enumerate(open_idx):
    ex = vqa_rad['train'][idx]
    axes[1, col].imshow(ex['image'], cmap='gray' if ex['image'].mode == 'L' else None)
    axes[1, col].axis('off')
    q = ex['question']
    q_wrap = '\n'.join([q[i:i+28] for i in range(0, len(q), 28)])
    ans = ex['answer']
    ans_short = ans[:22] + '...' if len(ans) > 22 else ans
    axes[1, col].set_title(f'Q: {q_wrap}\nA: {ans_short}',
                            fontsize=8, color='#2196F3')

plt.tight_layout()
fig_path = f'{BASE_DIR}/data/figures/exemples_vqarad.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure sauvegardée : {fig_path}')

## ÉTAPE 5 – Pipeline P1-Rad : CLAHE + Resize + Normalisation

In [ ]:
import cv2
import numpy as np
from PIL import Image

# ─────────────────────────────────────────────────────────
# Constantes ImageNet
# ─────────────────────────────────────────────────────────
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
TARGET_SIZE   = (448, 448)

# ─────────────────────────────────────────────────────────
# Paramètres CLAHE spécifiques VQA-RAD (radiologie N&B)
# Cl=3.0, tiles=8x8 – spec D1 du PDF projet
# ─────────────────────────────────────────────────────────
CLAHE_CLIP  = 3.0
CLAHE_TILES = (8, 8)


def to_rgb_robust(pil_img: Image.Image) -> Image.Image:
    """
    Convertit robustement une image PIL vers RGB.
    VQA-RAD contient : L (niveaux de gris), RGB, RGBA.
    """
    if pil_img.mode == 'RGB':
        return pil_img
    if pil_img.mode == 'RGBA':
        bg = Image.new('RGB', pil_img.size, (255, 255, 255))
        bg.paste(pil_img, mask=pil_img.split()[3])
        return bg
    # L, P, CMYK, YCbCr, 1
    return pil_img.convert('RGB')


def apply_clahe_rad(pil_img: Image.Image,
                    clip_limit: float = CLAHE_CLIP,
                    tile_grid: tuple  = CLAHE_TILES) -> Image.Image:
    """
    Applique CLAHE en espace LAB (canal L uniquement).
    Améliore le contraste local des radiographies sans saturer.
    Paramètre : Cl=3.0 (spec D1 PDF projet).
    """
    img_rgb = np.array(pil_img.convert('RGB'), dtype=np.uint8)
    # RGB → LAB
    img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l_chan, a_chan, b_chan = cv2.split(img_lab)
    # CLAHE sur canal L uniquement
    clahe   = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    l_clahe = clahe.apply(l_chan)
    # Fusionner et reconvertir en RGB
    img_lab_clahe = cv2.merge([l_clahe, a_chan, b_chan])
    img_rgb_out   = cv2.cvtColor(img_lab_clahe, cv2.COLOR_LAB2RGB)
    return Image.fromarray(img_rgb_out)


def resize_with_padding(pil_img: Image.Image,
                         target: tuple = TARGET_SIZE,
                         preserve_ratio: bool = True,
                         pad_color: tuple = (128, 128, 128)) -> Image.Image:
    """
    Resize 448×448 avec padding centré gris (preserve_ratio=True)
    ou stretch direct (preserve_ratio=False).
    """
    if not preserve_ratio:
        return pil_img.resize(target, Image.LANCZOS)
    w, h  = pil_img.size
    scale = min(target[0] / w, target[1] / h)
    nw, nh = int(w * scale), int(h * scale)
    pil_r  = pil_img.resize((nw, nh), Image.LANCZOS)
    canvas = Image.new('RGB', target, pad_color)
    canvas.paste(pil_r, ((target[0] - nw) // 2, (target[1] - nh) // 2))
    return canvas


def normalize_imagenet(img_array: np.ndarray) -> np.ndarray:
    """
    Normalisation ImageNet : /255 → (x − mean) / std.
    Retourne float32, shape (3, H, W) – format CHW pour PyTorch.
    """
    arr = img_array.astype(np.float32) / 255.0
    arr = (arr - IMAGENET_MEAN) / IMAGENET_STD
    return arr.transpose(2, 0, 1)   # HWC → CHW


def preprocess_rad(pil_img: Image.Image,
                   preserve_ratio: bool = True,
                   return_tensor: bool  = False) -> np.ndarray:
    """
    Pipeline P1-Rad complet (D1 – VQA-RAD) :
      1. PIL → RGB robuste  (gère L, RGBA, P, CMYK)
      2. RGB → LAB → CLAHE(Cl=3.0, tiles=8×8) → RGB
      3. Resize(448,448) + padding centré gris
      4. /255.0 → normalisation ImageNet → float32 CHW

    Args:
        pil_img       : Image PIL brute de VQA-RAD.
        preserve_ratio: Conserve le ratio avec padding. Défaut: True.
        return_tensor : Si True, retourne un torch.Tensor.

    Returns:
        np.ndarray float32 shape (3, 448, 448)  [ou torch.Tensor].
    """
    img_rgb    = to_rgb_robust(pil_img)
    img_clahe  = apply_clahe_rad(img_rgb)
    img_resize = resize_with_padding(img_clahe, TARGET_SIZE, preserve_ratio)
    arr        = normalize_imagenet(np.array(img_resize))
    if return_tensor:
        import torch
        return torch.from_numpy(arr)
    return arr


# ── Test rapide ──
sample_img = vqa_rad['train'][0]['image']
result = preprocess_rad(sample_img)
print('=== Test pipeline P1-Rad ===')
print(f'  Shape    : {result.shape}  (attendu: (3, 448, 448))')
print(f'  Dtype    : {result.dtype}  (attendu: float32)')
print(f'  Range    : [{result.min():.3f}, {result.max():.3f}]')
print(f'  Moy/canal: {result.mean(axis=(1,2)).round(3)}')
assert result.shape == (3, 448, 448), f'ERREUR shape: {result.shape}'
assert result.dtype == np.float32,    f'ERREUR dtype: {result.dtype}'
print('  Assertions OK ✓')

## ÉTAPE 6 – Visualisation CLAHE (avant / après)

In [ ]:
# Comparaison avant / après CLAHE sur 4 images radiologiques
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle(
    'VQA-RAD – CLAHE LAB (Cl=3.0, tiles=8×8) : Avant / Après\n'
    '(Canal L uniquement – contraste local amélioré)',
    fontsize=13, fontweight='bold'
)

sample_indices = [0, 10, 50, 100]

for col, idx in enumerate(sample_indices):
    ex     = vqa_rad['train'][idx]
    orig   = to_rgb_robust(ex['image'])
    clahed = apply_clahe_rad(orig)

    axes[0, col].imshow(orig)
    axes[0, col].axis('off')
    axes[0, col].set_title(f'Original #{idx}\n{orig.size[0]}×{orig.size[1]}', fontsize=9)

    axes[1, col].imshow(clahed)
    axes[1, col].axis('off')
    axes[1, col].set_title(f'CLAHE (Cl=3.0) #{idx}', fontsize=9)

plt.tight_layout()
fig_path = f'{BASE_DIR}/data/figures/clahe_comparaison_rad.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure sauvegardée : {fig_path}')

In [ ]:
# Histogramme du canal L (luminance) avant / après CLAHE
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('VQA-RAD – Histogramme canal L avant / après CLAHE (image #0)',
             fontsize=12, fontweight='bold')

ex0   = vqa_rad['train'][0]
orig0 = to_rgb_robust(ex0['image'])
clah0 = apply_clahe_rad(orig0)

# Extraire canal L
def get_l_channel(pil_img):
    arr = np.array(pil_img.convert('RGB'), dtype=np.uint8)
    lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB)
    return lab[:, :, 0]

l_orig = get_l_channel(orig0)
l_clah = get_l_channel(clah0)

axes[0].hist(l_orig.ravel(), bins=64, color='#607D8B', alpha=0.8)
axes[0].set_title('Original – Canal L')
axes[0].set_xlim(0, 255)
axes[0].set_xlabel('Intensité')
axes[0].set_ylabel('Pixels')

axes[1].hist(l_clah.ravel(), bins=64, color='#FF6F00', alpha=0.8)
axes[1].set_title('CLAHE (Cl=3.0) – Canal L')
axes[1].set_xlim(0, 255)
axes[1].set_xlabel('Intensité')

plt.tight_layout()
fig_path = f'{BASE_DIR}/data/figures/histogramme_clahe_rad.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure sauvegardée : {fig_path}')

In [ ]:
# Visualisation padding centré : original → CLAHE → 448×448
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle('VQA-RAD – Resize 448×448 avec padding centré gris',
             fontsize=13, fontweight='bold')

for col, idx in enumerate(sample_indices):
    ex      = vqa_rad['train'][idx]
    orig    = to_rgb_robust(ex['image'])
    clahed  = apply_clahe_rad(orig)
    resized = resize_with_padding(clahed)

    axes[0, col].imshow(orig)
    axes[0, col].axis('off')
    axes[0, col].set_title(f'Original #{idx}\n{orig.size[0]}×{orig.size[1]}', fontsize=9)

    axes[1, col].imshow(resized)
    axes[1, col].axis('off')
    axes[1, col].set_title('Resized 448×448', fontsize=9)

plt.tight_layout()
fig_path = f'{BASE_DIR}/data/figures/padding_comparaison_rad.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure sauvegardée : {fig_path}')

## ÉTAPE 7 – Test return_tensor (PyTorch)

In [ ]:
# Test return_tensor=True
print('── Test return_tensor=True ──')
try:
    import torch
    tensor = preprocess_rad(vqa_rad['train'][0]['image'], return_tensor=True)
    print(f'  torch.Tensor shape : {tensor.shape}')
    print(f'  dtype              : {tensor.dtype}')
    print(f'  Range              : [{tensor.min():.3f}, {tensor.max():.3f}]')
    print('  Tensor OK ✓')
except ImportError:
    print('  PyTorch non installé – test tensor ignoré')

## ÉTAPE 8 – Benchmark du Pipeline P1-Rad

In [ ]:
import time

print('=== Benchmark pipeline P1-Rad (200 images) ===')
N_BENCH = min(200, len(vqa_rad['train']))
times, errors = [], []

for idx in range(N_BENCH):
    try:
        img = vqa_rad['train'][idx]['image']
        t0  = time.perf_counter()
        _   = preprocess_rad(img)
        times.append(time.perf_counter() - t0)
    except Exception as e:
        errors.append((idx, str(e)))

times_ms = [t * 1000 for t in times]
print(f'  Images traitées : {len(times)}/{N_BENCH}')
print(f'  Erreurs         : {len(errors)}')
print(f'  Latence moyenne : {np.mean(times_ms):.1f} ms')
print(f'  Latence min/max : {np.min(times_ms):.1f} / {np.max(times_ms):.1f} ms')
print(f'  Médiane         : {np.median(times_ms):.1f} ms')
print(f'  Débit estimé    : {1000/np.mean(times_ms):.0f} images/s')

if errors:
    print('\n  Exemples d\'erreurs :')
    for idx, msg in errors[:3]:
        print(f'    idx={idx} : {msg}')

print('\n  Benchmark OK ✓')

## ÉTAPE 9 – Sauvegarde preprocess_rad.py

In [ ]:
script_content = '''
"""
preprocess_rad.py – D1 MMedAgent-Lite
Pipeline P1-Rad : CLAHE (Cl=3.0, LAB) + Resize(448,448) + Normalisation ImageNet
Spécifique : images radiologiques N&B (VQA-RAD)
"""

import cv2
import numpy as np
from PIL import Image
from datasets import load_dataset, load_from_disk
import time
import os

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
TARGET_SIZE   = (448, 448)
CLAHE_CLIP    = 3.0
CLAHE_TILES   = (8, 8)

def to_rgb_robust(pil_img):
    if pil_img.mode == \'RGB\': return pil_img
    if pil_img.mode == \'RGBA\':
        bg = Image.new(\'RGB\', pil_img.size, (255, 255, 255))
        bg.paste(pil_img, mask=pil_img.split()[3])
        return bg
    return pil_img.convert(\'RGB\')

def apply_clahe_rad(pil_img, clip_limit=CLAHE_CLIP, tile_grid=CLAHE_TILES):
    img_rgb = np.array(pil_img.convert(\'RGB\'), dtype=np.uint8)
    img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(img_lab)
    clahe   = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    l_eq    = clahe.apply(l)
    out     = cv2.cvtColor(cv2.merge([l_eq, a, b]), cv2.COLOR_LAB2RGB)
    return Image.fromarray(out)

def resize_with_padding(pil_img, target=TARGET_SIZE, preserve_ratio=True, pad_color=(128,128,128)):
    if not preserve_ratio: return pil_img.resize(target, Image.LANCZOS)
    w, h = pil_img.size
    scale = min(target[0]/w, target[1]/h)
    nw, nh = int(w*scale), int(h*scale)
    canvas = Image.new(\'RGB\', target, pad_color)
    canvas.paste(pil_img.resize((nw, nh), Image.LANCZOS), ((target[0]-nw)//2, (target[1]-nh)//2))
    return canvas

def normalize_imagenet(img_array):
    arr = img_array.astype(np.float32) / 255.0
    return ((arr - IMAGENET_MEAN) / IMAGENET_STD).transpose(2, 0, 1)

def preprocess_rad(pil_img, preserve_ratio=True, return_tensor=False):
    arr = normalize_imagenet(np.array(resize_with_padding(apply_clahe_rad(to_rgb_robust(pil_img)))))
    if return_tensor:
        import torch; return torch.from_numpy(arr)
    return arr

if __name__ == \'__main__\':
    print(\'=' * 55)
    print(\'  preprocess_rad.py – D1 MMedAgent-Lite\')
    print(\'=' * 55)
    vqa_rad = load_dataset(\'flaviagiammarino/vqa-rad\')
    print(f\'Train : {len(vqa_rad[\'train\'])}  |  Test : {len(vqa_rad[\'test\'])}\')
    sample = vqa_rad[\'train\'][0]
    result = preprocess_rad(sample[\'image\'])
    print(f\'  Shape  : {result.shape}\')
    print(f\'  dtype  : {result.dtype}\')
    print(f\'  Range  : [{result.min():.3f}, {result.max():.3f}]\')
    assert result.shape == (3, 448, 448)
    assert result.dtype == np.float32
    print(\'  Assertions OK ✓\')
    print(\'\\n  preprocess_rad.py OK – prêt pour D3 ✓\')
'''

script_path = f'{BASE_DIR}/dataset/scripts/preprocess_rad.py'
with open(script_path, 'w', encoding='utf-8') as f:
    f.write(script_content.strip())

print(f'Script sauvegardé : {script_path}')

## ÉTAPE 9 – Rapport Statistiques Final

In [ ]:
import json

rapport = {
    'dataset'     : 'flaviagiammarino/vqa-rad',
    'taille_go'   : '~0.3 Go',
    'splits'      : {'train': len(df_train), 'test': len(df_test), 'total': len(df_all)},
    'reponses'    : {
        'yes'     : int(n_yes),
        'no'      : int(n_no),
        'ouvertes': int(n_open),
        'pct_yes' : round(100*n_yes/len(df_train), 1),
        'pct_no'  : round(100*n_no/len(df_train), 1),
        'pct_open': round(100*n_open/len(df_train), 1),
    },
    'images'      : {
        'modes'      : df_train['img_mode'].value_counts().to_dict(),
        'width_mean' : round(df_train['img_width'].mean(), 1),
        'height_mean': round(df_train['img_height'].mean(), 1),
        'width_min'  : int(df_train['img_width'].min()),
        'width_max'  : int(df_train['img_width'].max()),
    },
    'questions'   : {
        'longueur_moy': round(df_train['q_len'].mean(), 2),
        'longueur_max': int(df_train['q_len'].max()),
        'top_mots'    : df_train['first_word'].value_counts().head(8).to_dict(),
    },
    'pipeline_p1_rad': {
        'etape_1': 'PIL → RGB robuste (gère L, RGBA, P)',
        'etape_2': 'RGB → LAB → CLAHE(clipLimit=3.0, tiles=8x8) → RGB',
        'etape_3': 'Resize(448,448) avec padding centré (preserve_ratio=True)',
        'etape_4': '/255.0 puis normalisation ImageNet (mean/std)',
        'output' : 'float32 shape (3, 448, 448) CHW',
    },
    'benchmark_ms': round(float(np.mean(times_ms)), 1) if times_ms else 'N/A',
    'figures'     : [
        'stats_vqarad.png',
        'categories_vqarad.png',
        'exemples_vqarad.png',
        'clahe_comparaison_rad.png',
        'histogramme_clahe_rad.png',
        'padding_comparaison_rad.png',
    ]
}

rapport_path = f'{BASE_DIR}/data/splits/rapport_stats_vqarad.json'
with open(rapport_path, 'w', encoding='utf-8') as f:
    json.dump(rapport, f, ensure_ascii=False, indent=2)

print('=' * 60)
print('  RAPPORT FINAL – VQA-RAD D1')
print('=' * 60)
print(f'  Dataset  : flaviagiammarino/vqa-rad')
print(f'  Taille   : ~0.3 Go')
print(f'  Train    : {len(df_train)}')
print(f'  Test     : {len(df_test)}')
print(f'  Total    : {len(df_all)}')
print()
print(f'  Réponses :')
print(f'    Yes     : {n_yes} ({100*n_yes/len(df_train):.1f}%)')
print(f'    No      : {n_no} ({100*n_no/len(df_train):.1f}%)')
print(f'    Ouvert  : {n_open} ({100*n_open/len(df_train):.1f}%)')
print()
print('  Pipeline P1-Rad :')
print('  1. PIL → RGB robuste (L, RGBA, P, CMYK → RGB)')
print('  2. RGB → LAB → CLAHE(clipLimit=3.0, tiles=8x8) → RGB')
print('  3. Resize(448,448) + padding centré gris (preserve_ratio)')
print('  4. /255.0 → normalisation ImageNet')
print()
print('  Fichiers produits sur Drive :')
for fig_name in rapport['figures']:
    print(f'    - {BASE_DIR}/data/figures/{fig_name}')
print(f'    - {script_path}')
print(f'    - {rapport_path}')
print()
print('NOTEBOOK D1 VQA-RAD TERMINÉ ✓')